In [31]:
import os
import re
import sys
import sys 
import numpy as np
import pandas as pd
import torch
from scipy.signal import find_peaks
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")   # non-interactive backend, avoids the inline backend bug
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline,Pipeline
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold, cross_val_score, GroupKFold
from sklearn.compose import ColumnTransformer

In [2]:
sys.path.append("/data1/yashvi_bhuva/anemia/PPG_BP/papagei-foundation-model")

import pyPPG.preproc as PP
from dotmap import DotMap
from linearprobing.utils import resample_batch_signal, load_model_without_module_prefix
from models.resnet import ResNet1DMoE

In [3]:
RAW_DIR = "ppg_data"              # folder of raw .txt files, e.g. "2_1.txt"
FS_ORIGINAL = 1000                 # PPG-BP native sampling rate — confirm
FS_TARGET_PAPAGEI = 125            # PaPaGei-S expected input rate
WEIGHTS_PATH = "papagei-foundation-model/weights/papagei_s.pt"
METADATA_PATH = "PPG-BP dataset.xlsx"

fname_pattern = re.compile(r"^(\d+)_(\d+)")


meta = pd.read_excel(METADATA_PATH, sheet_name="cardiovascular dataset", header=1)
meta["subject_ID"] = meta["subject_ID"].astype(int)
meta = meta.set_index("subject_ID")

# Features to be extracted from PPG

def get_peaks(ppg, fs):
    peaks, _ = find_peaks(ppg, distance=int(0.4 * fs))
    return peaks

def heart_rate_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    if len(peaks) < 2:
        return np.nan
    ppi = np.diff(peaks) / fs
    return 60 / np.mean(ppi)

def mean_peak_amplitude_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.mean(ppg[peaks]) if len(peaks) > 0 else np.nan

def std_peak_amplitude_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.std(ppg[peaks]) if len(peaks) > 0 else np.nan

def mean_peak_distance_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.mean(np.diff(peaks)) if len(peaks) >= 2 else np.nan

def extract_features(ppg, vpg, apg, fs):
    features = {
        "mean_ppg": np.mean(ppg), "std_ppg": np.std(ppg),
        "skew_ppg": stats.skew(ppg), "kurtosis_ppg": stats.kurtosis(ppg),
        "rms_ppg": np.sqrt(np.mean(ppg**2)), "max_ppg": np.max(ppg),
        "min_ppg": np.min(ppg), "range_ppg": np.ptp(ppg),
        "heart_rate": heart_rate_feature(ppg, fs),
        "mean_peak_amp": mean_peak_amplitude_feature(ppg, fs),
        "std_peak_amp": std_peak_amplitude_feature(ppg, fs),
        "mean_peak_distance": mean_peak_distance_feature(ppg, fs),
        "max_vpg": np.max(vpg), "min_vpg": np.min(vpg),
        "mean_vpg": np.mean(vpg), "std_vpg": np.std(vpg),
        "max_apg": np.max(apg), "min_apg": np.min(apg),
        "mean_apg": np.mean(apg), "std_apg": np.std(apg),
        "dominant_freq": np.nan, "spectral_energy": np.sum(ppg**2),
    }
    freqs = np.fft.rfftfreq(len(ppg), d=1/fs)
    fft_mag = np.abs(np.fft.rfft(ppg))
    if len(fft_mag) > 1:
        features["dominant_freq"] = freqs[1:][np.argmax(fft_mag[1:])]
    return features


# The below function cleans the raw PPG signal to match what PaPaGei was trained on
def preprocess_one_ppg_signal(waveform, frequency, fL=0.5, fH=12, order=4,
                                smoothing_windows={"ppg": 50, "vpg": 10, "apg": 10, "jpg": 10}):
    prep = PP.Preprocess(fL=fL, fH=fH, order=order, sm_wins=smoothing_windows)
    signal = DotMap()
    signal.v = waveform
    signal.fs = frequency
    signal.filtering = True
    ppg, ppg_d1, ppg_d2, ppg_d3 = prep.get_signals(signal)
    return ppg, ppg_d1, ppg_d2, ppg_d3


def is_signal_flat_lined_simple(sig, fs, flat_threshold=0.25, change_threshold=0.01, window_ms=20):
    sig = np.asarray(sig, dtype=float)
    sig_norm = (sig - np.mean(sig)) / (np.std(sig) + 1e-8)

    step = max(1, int(fs * window_ms / 1000))  # compare samples ~20ms apart, not adjacent
    diffs = np.abs(sig_norm[step:] - sig_norm[:-step])
    flat_fraction = np.sum(diffs < change_threshold) / len(diffs)
    return 1 if flat_fraction > flat_threshold else 0

In [5]:
# Creating the PaPaGei model and loading weights
model_config = {
    'base_filters': 32, 'kernel_size': 3, 'stride': 2, 'groups': 1,
    'n_block': 18, 'n_classes': 512, 'n_experts': 3
}
papagei_model = ResNet1DMoE(in_channels=1, **model_config)
papagei_model = load_model_without_module_prefix(papagei_model, WEIGHTS_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
papagei_model.to(device).eval()

cuda


ResNet1DMoE(
  (first_block_conv): MyConv1dPadSame(
    (conv): Conv1d(1, 32, kernel_size=(3,), stride=(1,))
  )
  (first_block_bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (first_block_relu): ReLU()
  (basicblock_list): ModuleList(
    (0): BasicBlock(
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu1): ReLU()
      (do1): Dropout(p=0.5, inplace=False)
      (conv1): MyConv1dPadSame(
        (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
      )
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu2): ReLU()
      (do2): Dropout(p=0.5, inplace=False)
      (conv2): MyConv1dPadSame(
        (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
      )
      (max_pool): MyMaxPool1dPadSame(
        (max_pool): MaxPool1d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
      )
    )
    

In [6]:
# Preprocess all raw PPG files, compute VPG& APG, extract features, pair with BP labels and save to CSV
log_rows = []
feature_rows = []
n_ok, n_fail, n_flatlined = 0, 0, 0

for file in os.listdir(RAW_DIR):
    try:
        match = fname_pattern.match(file)
        if not match:
            raise ValueError(f"Could not parse subject_id/segment from '{file}'")
        subject_id, segment = int(match.group(1)), int(match.group(2))
        if subject_id not in meta.index:
            raise KeyError(f"subject_id {subject_id} not in metadata")

        raw_ppg = np.loadtxt(os.path.join(RAW_DIR, file))

        # -- filter + derivatives (PaPaGei's exact preprocessing) --
        ppg_filt, vpg_filt, apg_filt, _ = preprocess_one_ppg_signal(
            waveform=raw_ppg, frequency=FS_ORIGINAL
        )

        # -- quality gate --
        if is_signal_flat_lined_simple(ppg_filt, fs=FS_ORIGINAL):
            n_flatlined += 1
            log_rows.append({"file": file, "status": "flatlined", "error": ""})
            continue

        # -- hand-crafted features (on the same filtered signal) --
        feature_row = extract_features(ppg_filt, vpg_filt, apg_filt, FS_ORIGINAL)

        # -- PaPaGei embedding --
        resampled = resample_batch_signal(
            ppg_filt[np.newaxis, :], fs_original=FS_ORIGINAL,
            fs_target=FS_TARGET_PAPAGEI, axis=-1
        )
        signal_tensor = torch.Tensor(resampled).unsqueeze(dim=1).to(device)
        with torch.inference_mode():
            outputs = papagei_model(signal_tensor)
            embedding = outputs[0].cpu().numpy().squeeze()
        for i, v in enumerate(embedding):
            feature_row[f"papagei_{i}"] = v

        # -- labels / demographics --
        label_row = meta.loc[subject_id]
        feature_row["SBP"] = label_row["Systolic Blood Pressure(mmHg)"]
        feature_row["DBP"] = label_row["Diastolic Blood Pressure(mmHg)"]
        feature_row["Age"] = label_row["Age(year)"]
        feature_row["BMI"] = label_row["BMI(kg/m^2)"]
        feature_row["Hypertension"] = label_row["Hypertension"]

        feature_row["file"] = file
        feature_row["subject_id"] = subject_id
        feature_row["segment"] = segment

        feature_rows.append(feature_row)
        log_rows.append({"file": file, "status": "ok", "error": ""})
        n_ok += 1

    except Exception as e:
        log_rows.append({"file": file, "status": "failed", "error": str(e)})
        n_fail += 1
        print(f"[FAILED] {file}: {e}")

print(f"\nDone. {n_ok} ok, {n_flatlined} flatlined (excluded), {n_fail} failed.")
features_df = pd.DataFrame(feature_rows)
log_df = pd.DataFrame(log_rows)
print(features_df.shape)



Done. 657 ok, 0 flatlined (excluded), 0 failed.
(657, 542)


In [ ]:
# Checking how features are related to DBP using Ridge regression with GroupKFold cross-validation
data = features_df.dropna(subset=["DBP"]).copy()

# Target
y = data["DBP"].values

# Subject IDs for leakage-free splitting
groups = data["subject_id"].values
gkf = GroupKFold(n_splits=5)  #Subject-wise splitting to avoid data leakage

conditions = {
    "Demographics only":
        ["BMI", "Age"],
    "Hand-crafted PPG feats":
        hand_crafted_cols + ["BMI", "Age"],
    "PaPaGei embeddings":
        papagei_cols + ["BMI", "Age"],
    "All features":
        hand_crafted_cols + papagei_cols + ["BMI", "Age"]
        }

for name, cols in conditions.items():
    print("\n", name)
    X = data[cols].values
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )
    r2_scores = cross_val_score(
        model,
        X,
        y,
        cv=gkf,
        groups=groups,
        scoring="r2"
    )
    print(
        f"R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}"
    )


 Demographics only
R² = -0.012 ± 0.090

 Hand-crafted PPG feats
R² = 0.012 ± 0.100

 PaPaGei embeddings
R² = -0.486 ± 0.182

 All
R² = -0.523 ± 0.203


In [17]:
# Checking how features are related to SBP using Ridge regression with GroupKFold cross-validation
data = features_df.dropna(subset=["SBP"]).copy()

# Target
y = data["SBP"].values

# Subject IDs for leakage-free splitting
groups = data["subject_id"].values
gkf = GroupKFold(n_splits=5)  #Subject-wise splitting to avoid data leakage

conditions = {
    "Demographics only":
        ["BMI", "Age"],
    "Hand-crafted PPG feats":
        hand_crafted_cols + ["BMI", "Age"],
    "PaPaGei embeddings":
        papagei_cols + ["BMI", "Age"],
    "All features":
        hand_crafted_cols + papagei_cols + ["BMI", "Age"]
        }

for name, cols in conditions.items():
    print("\n", name)
    X = data[cols].values
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )
    r2_scores = cross_val_score(
        model,
        X,
        y,
        cv=gkf,
        groups=groups,
        scoring="r2"
    )
    print(
        f"R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}"
    )


 Demographics only
R² = 0.138 ± 0.151

 Hand-crafted PPG feats
R² = 0.141 ± 0.155

 PaPaGei embeddings
R² = -0.459 ± 0.392

 All features
R² = -0.589 ± 0.571


In [ ]:
# Checking how features are related to hypertension status using Ridge regression with GroupKFold cross-validation
hypertension_map = {
    "Normal": 0,
    "Prehypertension": 1,
    "Stage 1 hypertension": 2,
    "Stage 2 hypertension": 3
}

features_df["Hypertension_class"] = (
    features_df["Hypertension"]
    .map(hypertension_map)
)
# print(features_df["Hypertension_class"].value_counts())

def evaluate_hypertension():
    data = features_df.dropna(
        subset=["Hypertension_class"]
    ).copy()

    y = data["Hypertension_class"].values
    groups = data["subject_id"].values
    conditions = {
        "Demographics only":
            ["BMI", "Age"],
        "Hand-crafted PPG feats":
            hand_crafted_cols + ["BMI", "Age"],
        "PaPaGei embeddings":
            papagei_cols + ["BMI", "Age"],
        "All features":
            hand_crafted_cols + papagei_cols + ["BMI", "Age"]
    }
    gkf = GroupKFold(n_splits=5)
    for name, cols in conditions.items():

        X = data[cols].values


        model = make_pipeline(
            StandardScaler(),

            LogisticRegression(
                max_iter=2000,
                multi_class="multinomial",
                solver="lbfgs"
            )
        )


        accuracy = cross_val_score(
            model,
            X,
            y,
            cv=gkf,
            groups=groups,
            scoring="f1_macro"
        )


        print(
            f"{name}: "
            f"Accuracy = {accuracy.mean():.3f} ± {accuracy.std():.3f}"
        )
evaluate_hypertension()

Demographics only: Accuracy = 0.326 ± 0.100
Hand-crafted PPG feats: Accuracy = 0.294 ± 0.053
PaPaGei embeddings: Accuracy = 0.336 ± 0.045
All features: Accuracy = 0.350 ± 0.033


In [36]:
from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA

# Try much stronger regularization, tuned automatically per fold
alphas = [0.1, 1, 10, 100, 1000, 10000]
groups = data["subject_id"].values
# Just stronger Ridge alpha, no dimensionality reduction
X = data[papagei_cols + ["BMI", "Age"]].values
pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas))
r2 = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring="r2")
print(f"PaPaGei + RidgeCV (auto-tuned alpha): R² = {r2.mean():.3f} ± {r2.std():.3f}")

# PCA down to far fewer components first, THEN Ridge
for n_comp in [5, 10, 20, 50, 75, 100, 150, 200]:
    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=n_comp),
        RidgeCV(alphas=alphas)
    )
    r2 = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring="r2")
    print(f"PaPaGei PCA({n_comp}) + RidgeCV: R² = {r2.mean():.3f} ± {r2.std():.3f}")

PaPaGei + RidgeCV (auto-tuned alpha): R² = 0.035 ± 0.087
PaPaGei PCA(5) + RidgeCV: R² = -0.012 ± 0.058
PaPaGei PCA(10) + RidgeCV: R² = -0.017 ± 0.048
PaPaGei PCA(20) + RidgeCV: R² = -0.020 ± 0.063
PaPaGei PCA(50) + RidgeCV: R² = 0.035 ± 0.102
PaPaGei PCA(75) + RidgeCV: R² = 0.030 ± 0.093
PaPaGei PCA(100) + RidgeCV: R² = 0.032 ± 0.086
PaPaGei PCA(150) + RidgeCV: R² = 0.032 ± 0.087
PaPaGei PCA(200) + RidgeCV: R² = 0.035 ± 0.087


In [ ]:
data = features_df.dropna(subset=["DBP"]).copy()

y = data["DBP"].values
groups = data["subject_id"].values


papagei_features = papagei_cols

other_features = (
    hand_crafted_cols +
    ["BMI", "Age"]
)


X = data[papagei_features + other_features]


# PCA only on Papagei embeddings
preprocessor = ColumnTransformer(

    transformers=[

        (
            "papagei",
            Pipeline([
                ("scale", StandardScaler()),
                ("pca", PCA(n_components=50))
            ]),
            papagei_features
        ),


        (
            "other",
            StandardScaler(),
            other_features
        )
    ]
)
model = Pipeline([
    ("preprocess", preprocessor),
    ("ridge",
     RidgeCV(
        alphas=[0.1,1,10,100,1000,10000]
     ))
])

gkf = GroupKFold(n_splits=5)

r2_scores = cross_val_score(
    model,
    X,
    y,
    cv=gkf,
    groups=groups,
    scoring="r2"
)

print(
    f"PaPaGei PCA(50) + Handcrafted + Demographics: "
    f"R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}"
)


PaPaGei PCA(50) + Handcrafted + Demographics: R² = 0.037 ± 0.100


## Neural Network ##

In [43]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

In [68]:
data = features_df.dropna(subset=["DBP"]).copy()

feature_cols = (
    papagei_cols 
    # hand_crafted_cols +
    # ["BMI", "Age"]
)

X = data[feature_cols].values


groups = data["subject_id"].values


y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1,1)
)

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1,1)
)

In [64]:
class BPNet(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,1)
        )


    def forward(self,x):
        return self.network(x)

In [69]:
class BPNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),


            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),


            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),


            nn.Linear(128, 64),
            nn.ReLU(),


            nn.Linear(64, 1)
        )


    def forward(self, x):
        return self.network(x)

In [70]:
def train_model(X_train, y_train, X_val, y_val):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Training on:", device)


    model = BPNet(
        input_dim=X_train.shape[1]
    ).to(device)



    criterion = nn.MSELoss()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=10
    )


    X_train = torch.tensor(
        X_train,
        dtype=torch.float32
    ).to(device)


    y_train = torch.tensor(
        y_train,
        dtype=torch.float32
    ).view(-1,1).to(device)



    X_val = torch.tensor(
        X_val,
        dtype=torch.float32
    ).to(device)


    y_val = torch.tensor(
        y_val,
        dtype=torch.float32
    ).view(-1,1).to(device)



    epochs = 100


    best_loss = np.inf
    patience = 25
    counter = 0


    for epoch in range(epochs):

        # -----------------
        # Training
        # -----------------

        model.train()

        optimizer.zero_grad()


        pred = model(X_train)


        train_loss = criterion(
            pred,
            y_train
        )


        train_loss.backward()

        optimizer.step()



        # -----------------
        # Validation
        # -----------------

        model.eval()

        with torch.no_grad():

            val_pred = model(X_val)

            val_loss = criterion(
                val_pred,
                y_val
            )


        scheduler.step(val_loss)



        # print every epoch
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss.item():.4f} "
            f"Val Loss: {val_loss.item():.4f}"
        )


        # Early stopping
        if val_loss < best_loss:

            best_loss = val_loss
            counter = 0

            best_model = model.state_dict()


        else:

            counter += 1


        if counter >= patience:

            print("Early stopping")
            break



    model.load_state_dict(best_model)


    model.eval()

    with torch.no_grad():

        predictions = model(X_val).cpu().numpy()


    return predictions

In [71]:
gkf = GroupKFold(n_splits=5)


r2_scores = []
mae_scores = []


for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups)):


    print("Fold:", fold+1)


    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]


    # normalize only using training data

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)



    predictions = train_model(
        X_train,
        y_train,
        X_test,
        y_test
    )


    r2 = r2_score(
        y_test,
        predictions
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )


    r2_scores.append(r2)
    mae_scores.append(mae)


    print(
        "R2:",
        round(r2,3),
        "MAE:",
        round(mae,3)
    )



print("\nFinal Results")

print(
    "R2:",
    np.mean(r2_scores),
    "+/-",
    np.std(r2_scores)
)

print(
    "MAE:",
    np.mean(mae_scores)
)

Fold: 1
Training on: cuda
Epoch [1/100] Train Loss: 5305.0771 Val Loss: 5210.6499
Epoch [2/100] Train Loss: 5303.3345 Val Loss: 5210.4482
Epoch [3/100] Train Loss: 5299.9434 Val Loss: 5210.1304
Epoch [4/100] Train Loss: 5298.3872 Val Loss: 5209.6978
Epoch [5/100] Train Loss: 5296.1602 Val Loss: 5209.1743
Epoch [6/100] Train Loss: 5293.5654 Val Loss: 5208.4814
Epoch [7/100] Train Loss: 5292.8730 Val Loss: 5207.6689
Epoch [8/100] Train Loss: 5289.4888 Val Loss: 5206.7119
Epoch [9/100] Train Loss: 5287.9365 Val Loss: 5205.5591
Epoch [10/100] Train Loss: 5284.9780 Val Loss: 5204.2056
Epoch [11/100] Train Loss: 5283.8789 Val Loss: 5202.6567
Epoch [12/100] Train Loss: 5281.2544 Val Loss: 5200.9102
Epoch [13/100] Train Loss: 5278.5435 Val Loss: 5198.9814
Epoch [14/100] Train Loss: 5276.0830 Val Loss: 5196.8091
Epoch [15/100] Train Loss: 5273.9365 Val Loss: 5194.4688
Epoch [16/100] Train Loss: 5271.1221 Val Loss: 5191.9980
Epoch [17/100] Train Loss: 5268.9395 Val Loss: 5189.4141
Epoch [18/100]

In [ ]:
# Creating one representative sample per subject by aaveraging all segments for that subject
hand_crafted_cols = ["mean_ppg","std_ppg","skew_ppg","kurtosis_ppg","rms_ppg","max_ppg",
    "min_ppg","range_ppg","heart_rate","mean_peak_amp","std_peak_amp","mean_peak_distance",
    "max_vpg","min_vpg","mean_vpg","std_vpg","max_apg","min_apg","mean_apg","std_apg",
    "dominant_freq","spectral_energy"]
papagei_cols = [c for c in features_df.columns if c.startswith("papagei_")]

subject_df = features_df.groupby("subject_id")[hand_crafted_cols + papagei_cols].mean().reset_index()
labels = features_df.drop_duplicates("subject_id")[["subject_id","SBP","DBP","Age","BMI","Hypertension"]]
subject_df = subject_df.merge(labels, on="subject_id")
print(subject_df.shape)